# Pipeline diário — Negociação Secundária de Crédito Privado

Este notebook é o **uso do dia a dia**. Antes dele, rode **uma vez** os dois notebooks de setup:
`setup_1_teste.ipynb` (testa cada fluxo) → `setup_2_carga.ipynb` (carga histórica da base).

Cada fluxo é uma função em `scripts/pipeline_core.py` (`pc.boletim`, `pc.ntnb`, `pc.calc_taxa`, ...) que chama o CLI do script. **O CLI vive só no `pipeline_core`** — o `run_diario.py` (Task Scheduler) chama exatamente as mesmas funções, então mudar um fluxo aqui vale para os dois. Nada aborta o notebook: cada passo mostra `[OK]`/`[FALHA]`.

**Rode a partir da pasta `code/`.** Seções: **A)** testar/rodar um fluxo isolado · **B)** rotina diária (últimos `N_DIAS` dias úteis, data ajustável).

## Config (rodar primeiro)

In [ ]:
import sys
from pathlib import Path

scripts = Path.cwd() / "scripts"
if not scripts.exists():
    raise SystemExit(f"Rode o notebook a partir da pasta code/. cwd atual: {Path.cwd()}")
sys.path.insert(0, str(scripts))
import pipeline_core as pc

# Datas de teste (dia util mais recente e o anterior). Ajuste se quiser.
X    = pc.ultimos_n_dias_uteis(1)[0].isoformat()
Xant = pc.dia_util_anterior(X).isoformat()
print("Data de teste X =", X, "| X-1u =", Xant)

# A) Testar / rodar um fluxo isolado
Útil para debug (proxy, Playwright, login) ou para rerodar só uma fonte. Cada bloco chama uma função de fluxo do `pipeline_core`.

### Scraping (rede)

In [ ]:
pc.boletim(Xant, X)          # 1. Boletim B3 (negocios) — X-1u e X

In [ ]:
pc.anbima_deb(X)             # 2. Anbima debentures (taxa indicativa)

In [ ]:
pc.anbima_cricra(X)          # 3. Anbima CRI/CRA — Playwright

In [ ]:
pc.fianalytics()             # 4. FI Analytics planilha — Playwright + login

In [ ]:
pc.anbima_data(Xant, X)      # 5. Anbima Data (caracteristicas + fluxo) — Playwright

In [ ]:
pc.ntnb(Xant, X)             # 6. Anbima NTN-B (MtM) — X-1u e X

In [ ]:
pc.curva_di(Xant); pc.curva_di(X)   # 7. Curva DI B3 (MtM) — X-1u e X

In [ ]:
pc.outstanding(X)            # 8. Outstanding Bloomberg — SO NO BANCO

### Cálculo (local — sem rede)

In [ ]:
pc.calc_taxa(X)              # 9. Taxa por trade (FI Analytics -> B3) — ANTES de filtrar

In [ ]:
pc.filtrar(X)                # 10. Filtrar (VALIDO / FUNDO / BROKER / PF)

In [ ]:
pc.spread_anbima(X)          # 11. Spread Anbima das indicativas

In [ ]:
pc.match_ref()               # 12. Match de referencia (global) — ANTES de spread_over

In [ ]:
pc.spread_over(X)            # 13. Spread over dos trades

In [ ]:
pc.relatorio()               # 14. Gerar relatorio (toda a base)

# B) Rotina diária
Roda a cadeia completa para os últimos `N_DIAS` dias úteis (pega correções retroativas) e gera o relatório no fim. É o que o `run_diario.py` chama no Task Scheduler.

In [ ]:
N_DIAS = 5   # quantos dias uteis reprocessar
pc.run_ultimos_n(N_DIAS)

Rodar um único dia de liquidação inteiro:

In [ ]:
DIA = X   # ajuste a data de liquidacao
pc.run_dia(DIA)